In [1]:
import cv2
import mediapipe as mp
import numpy as np
from inference import get_model
import supervision as sv
from collections import deque
import os

In [2]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

I0000 00:00:1736512992.329655 17077215 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 83), renderer: Apple M2


In [3]:
model = get_model(model_id="ball_detection-onors/3", api_key="YOUR_ROBOFLOW_API_KEY")

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1736512992.441887 18772518 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1736512992.460121 18772517 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
2025-01-10 19:43:14.874712 [W:onnxruntime:, coreml_execution_provider.cc:107 GetCapability] CoreMLExecutionProvider::GetCapability, number of partitions supported by CoreML: 11 number of nodes in the graph: 233 number of nodes supported by CoreML: 221


In [4]:
def calculate_angle(a, b, c):
    a = np.array(a)  # First point
    b = np.array(b)  # Midpoint
    c = np.array(c)  # End point

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360.0 - angle

    return angle

In [5]:
# def detect_basketball(frame):
#     # Resize frame for Roboflow input
#     input_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#     predictions = model.predict(input_image).json()  # Get predictions
#     detections = predictions['predictions']
#     # Find the basketball
#     for detection in detections:
#         if detection['class'] == 'basketball':
#             x_center = int(detection['x'])
#             y_center = int(detection['y'])
#             return (x_center, y_center)
#     return None

In [6]:
def detect_jump(landmarks):
    left_foot = landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y
    right_foot = landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y
    ground_level = max(left_foot, right_foot)
    return left_foot < ground_level - 0.02 and right_foot < ground_level - 0.02

In [7]:
def analyze_pose(landmarks):
    left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
    left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                 landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
    left_ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
                  landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]

    knee_angle = calculate_angle(left_hip, left_knee, left_ankle)

    squared_up = 85 <= knee_angle <= 95  # Adjust range for squaring up
    return squared_up

In [8]:
def detect_release(ball_detections, landmarks):
    if not ball_detections:
        return False

    left_wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
                  landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]
    right_wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
                   landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

    for detection in ball_detections:
        ball_center = [(detection[0] + detection[2]) / 2, (detection[1] + detection[3]) / 2]
        if np.linalg.norm(np.array(ball_center) - np.array(left_wrist)) < 0.05 or \
           np.linalg.norm(np.array(ball_center) - np.array(right_wrist)) < 0.05:
            return False

    return True

In [14]:
video_path = "good.mp4"
cap = cv2.VideoCapture(video_path)

squared_up_frame = None
release_frame = None
ball_last_detected = True

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = pose.process(rgb_frame)

    results = model.infer(frame)[0]
    detections = sv.Detections.from_inference(results)
    ball_detected = len(detections) > 0

    if result.pose_landmarks:
        landmarks = result.pose_landmarks.landmark

        if analyze_pose(landmarks) and squared_up_frame is None:
            squared_up_frame = frame.copy()
            cv2.imwrite("results/squared_up_frame.jpg", squared_up_frame)

        if not ball_detected and ball_last_detected:
            release_frame = frame.copy()
            cv2.imwrite("results/release_frame.jpg", release_frame)

    ball_last_detected = ball_detected

    cv2.imshow('Frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [15]:
print("Feedback Results:")
print(f"Squared Up: {'Captured' if squared_up_frame is not None else 'Not Captured'}")
print(f"Release: {'Captured' if release_frame is not None else 'Not Captured'}")

Feedback Results:
Squared Up: Captured
Release: Captured
